# Mypy: Type Checking for Engineering Code

## The Core Question

**How do you catch unit errors and type bugs before runtime?**

In engineering code, a `float` where a `str` is expected — or a missing unit conversion — can be catastrophic. A beam-deflection function that silently accepts `"500"` (a string) instead of `500.0` (newtons) won't crash; it will just give you a nonsense deflection, and you'll find out when the structure fails.

Mypy catches these **at check time**, before the code ever runs.

## Why this library?

Mypy is a static type checker. You annotate your functions with types (`def f(x: float) -> float`), and mypy reads the code *without running it* to find mismatches. Unlike a runtime test, it flags the bug the moment you write it — no need to construct a failing input. For engineering code, that means: *a `mass = "10"` (string) is caught before it breaks your FEA, not after.*

## The Workflow

```
  write function + type hints
        |
        v
  run:  mypy script.py
        |
        +--> no errors  -->  types are consistent, safe to run
        +--> errors      -->  fix the mismatch NOW (it's a one-line fix)
```


In [ ]:
import mypy.version
print('mypy', mypy.version.__version__)

## 1. A Function With Type Hints

Type hints document the **contract** of a function: what it accepts and what it returns. For a beam-deflection calculator, the contract is clear — lengths in metres, loads in newtons, flexural rigidity in N·m², deflection in metres.

```python
def max_deflection(length_m: float, load_n: float, ei: float) -> float:
```

The hints are the *specification*. Mypy then checks that every call site honours it.


In [ ]:
def max_deflection(length_m: float, load_n: float, ei: float) -> float:
    """Max deflection of a simply-supported beam under a central point load.

    delta = L^3 * P / (48 * E * I)
    """
    return length_m**3 * load_n / (48.0 * ei)

# A correct call — mypy is happy with this
print('deflection (m):', max_deflection(2.0, 500.0, 2.1e7))

## 2. Introduce Type Errors

Now we write a script with **three deliberate bugs** — the kind that sneak into real code:

1. **Wrong type argument** — passing `"500"` (a string) where a `float` load is expected. This is the classic *unit/typing* bug: someone pasted a value from a spreadsheet as text.
2. **Wrong return type** — a function promised to return a `float` but returns a `str`.
3. **Missing argument** — calling with two arguments where three are required.

None of these crash at import time. Mypy finds all three.


In [ ]:
# beam_errors.py  (the script mypy will check)
def max_deflection(length_m: float, load_n: float, ei: float) -> float:
    return length_m**3 * load_n / (48.0 * ei)

# Error 1: passing a string where a float is expected (unit/typing bug)
bad = max_deflection(2.0, "500", 2.1e7)

# Error 2: returning a string where a float is expected
def stiffness(ei: float) -> float:
    return "2.1e7"  # should be a float, not a string

# Error 3: calling with wrong number of arguments
missing = max_deflection(2.0, 500.0)

## 3. Run Mypy

Mypy reads the file and reports every type mismatch, with the line number and a short explanation.


In [ ]:
import subprocess, sys, tempfile, os

script = '''
def max_deflection(length_m: float, load_n: float, ei: float) -> float:
    return length_m**3 * load_n / (48.0 * ei)

bad = max_deflection(2.0, "500", 2.1e7)

def stiffness(ei: float) -> float:
    return "2.1e7"

missing = max_deflection(2.0, 500.0)
'''

with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False, encoding='utf-8') as f:
    f.write(script)
    path = f.name

result = subprocess.run([sys.executable, '-m', 'mypy', path],
                        capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
os.unlink(path)

### Reading the output

```
beam_errors.py:13: error: Argument 2 to "max_deflection" has incompatible type "str"; expected "float"  [arg-type]
beam_errors.py:17: error: Incompatible return value type (got "str", expected "float")  [return-value]
beam_errors.py:20: error: Missing positional argument "ei" in call to "max_deflection"  [call-arg]
Found 3 errors in 1 file (checked 1 source file)
```

Three bugs, three precise messages, **zero lines of code executed**. Each message tells you the line, the offending type, and the expected type.

**Engineering takeaway:** mypy turns "why is my deflection `nan`?" into "line 13: you passed a string." The first is a debugging session; the second is a one-line fix. For code that feeds a 1000-hour simulation, that difference is everything.


## 4. Exercises

1. **Annotate your own code.** Take a heat-transfer function you've written and add type hints to every parameter and return. Run `mypy` on it. How many errors did you find?

2. **The unit trap.** Write `def area(length_m: float, width_m: float) -> float`. Call it as `area(3, "4")`. What does mypy say? Now call it as `area(3, 4)` (ints) — does mypy complain? (Hint: ints are a subtype of float.)

3. **Optional inputs.** Add a parameter `deflection_limit: float | None = None` that, when given, returns a boolean "is the deflection within limit?" Run mypy and make sure the `None` case is handled.

4. **Strict mode.** Re-run with `mypy --strict beam_errors.py`. What *additional* errors appear? (Strict mode flags missing annotations, untyped defs, and more.)
